<a href="https://colab.research.google.com/github/DuncanBdb/gdp-dashboard/blob/main/beam/beam-batch-streaming-pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Batch and Streaming Pipelines with Apache Beam

Notebook: `beam-batch-streaming-pipeline.ipynb`
Data: an online retailer's customers (`users_v.csv`) and orders (`orders_v.csv`)
Platform: Google Colab, Apache Beam Python SDK, local runner

Your course handout gives the marks for each task, how to submit, and which track you complete.

### Structure

| Part | Covers | Tasks |
| :--- | :--- | :--- |
| Core (everyone) | Composite transforms and sharded output; combining; batch and windowed counts as one model; reading the pipeline graph | 1–3 |
| Track A: Joins | Joining two datasets with `CoGroupByKey`; a live-dashboard design decision | 4 |
| Track B: Late data | Event time, watermarks, allowed lateness, triggers and accumulation; a lateness-policy design decision | 5 |

Complete the core, then the track your handout assigns.

### Ground rules

* Do not change the pinned Beam version. The expected answers assume it.
* Every answer must come from a Beam pipeline. Do not compute answers with pandas or with plain Python loops over the files.
* The helpers in Section 0 handle the string processing (dates, addresses, CSV lines), so your work is the pipeline itself.
* Do not commit the data (`*.csv`), the `output/` folder, or cell outputs. Export your evidence PDF before clearing outputs (see the final section).

### Before you start

If you are new to Beam, work through the official introductory notebooks first. They open directly in Colab:

1. [Try Apache Beam (Python)](https://colab.research.google.com/github/apache/beam/blob/master/examples/notebooks/get-started/try-apache-beam-py.ipynb): a minimal word-count pipeline.
2. [Learn Beam basics by doing](https://colab.research.google.com/github/apache/beam/blob/master/examples/notebooks/get-started/learn_beam_basics_by_doing.ipynb): `PCollection`, `PTransform`, `Map`, `FlatMap`, `Filter` and `Combine`.
3. Optional, before Task 3 and Track B: [Learn Beam windowing by doing](https://colab.research.google.com/github/apache/beam/blob/master/examples/notebooks/get-started/learn_beam_windowing_by_doing.ipynb): fixed, sliding and session windows on batch data.

## 0. Environment setup

Install the pinned Apache Beam release. If Colab asks you to restart the session after the install, restart it and run the notebook again from the top. A restart keeps the installed packages and the files in `/content`, so the second install finds Beam already in place.

If the runtime disconnects instead (for example after a period of inactivity, or through Runtime > Disconnect and delete runtime), Colab assigns a new virtual machine and everything in `/content` is lost, including `data/` and `output/`. Run the notebook again from the top in that case as well; every cell is safe to re-run.

In [1]:
# Pinned version: do not change
BEAM_VERSION = "2.77.0"

%pip install -q apache-beam=={BEAM_VERSION}

import apache_beam as beam
print(f"Apache Beam version: {beam.__version__}")
assert beam.__version__ == BEAM_VERSION, (
    f"Expected Beam {BEAM_VERSION} but found {beam.__version__}. "
    "Restart the runtime (Runtime > Restart session) and re-run this cell."
)

Apache Beam version: 2.77.0


### Data

The cell below downloads both files once and verifies their checksums, so that everyone works on identical data.

In [2]:
import hashlib
import os
import urllib.request

DATA = {
    "users_v.csv": ("https://storage.googleapis.com/bdt-beam/users_v.csv",
                    "10b216aa1cc982affd33675448a0770470e3b6baf25b8fa4be53273acc9937c4"),
    "orders_v.csv": ("https://storage.googleapis.com/bdt-beam/orders_v.csv",
                     "2e2826f5416affd5cc89f635f6365f61829e30dd023a2ee6082f5ffa2687b6c4"),
}
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

for name, (url, sha256) in DATA.items():
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print(f"Downloading {name}")
        urllib.request.urlretrieve(url, path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == sha256, f"Checksum mismatch for {name}: delete {path} and re-run this cell."
    print(f"{name}: {os.path.getsize(path) / 1e6:.1f} MB, checksum verified")

USERS_FILE = os.path.join(DATA_DIR, "users_v.csv")
ORDERS_FILE = os.path.join(DATA_DIR, "orders_v.csv")
!head -n 3 {USERS_FILE}
!head -n 3 {ORDERS_FILE}

users_v.csv: 0.1 MB, checksum verified
orders_v.csv: 37.8 MB, checksum verified
user_id,name,gender,age,address,date_joined
1,Anthony Wolf,male,73,New Rachelburgh-VA-49583,2019/03/13
2,James Armstrong,male,56,North Jillianfort-UT-86454,2020/11/06
order_no,user_id,product_list,date_purchased
1000,1887,Cassava,2000-01-01
1001,838,"Calabash, Water Spinach",2000-01-01


### Provided helpers

Run this cell without changing it.

* `new_pipeline()` creates a `beam.Pipeline`. Use it instead of `beam.Pipeline()`: it stops Beam from parsing the notebook kernel's own command-line arguments.
* Batch pipelines run on Beam's local runner, which in this release is the Prism runner; log lines that mention Prism are expected. Warnings about `GCSFileSystem` or `apitools` can be ignored, because the notebook uses no Google Cloud services.

In [3]:
import csv
import datetime as dt
import glob
import shutil

from apache_beam.options.pipeline_options import PipelineOptions

OUT_DIR = "output"
shutil.rmtree(OUT_DIR, ignore_errors=True)   # start every run of the notebook from a clean output folder
os.makedirs(OUT_DIR)

DAY = 24 * 60 * 60          # seconds
WINDOW_DAYS = 30
AGE_BANDS = [(18, 30), (31, 45), (46, 60), (61, 80)]


def new_pipeline(**options):
    # A Pipeline that ignores the kernel's command-line arguments, e.g. new_pipeline(streaming=True)
    return beam.Pipeline(options=PipelineOptions(flags=[], **options))


def to_iso_date(date_joined):
    # '2019/03/13' -> '2019-03-13'
    return dt.datetime.strptime(date_joined, "%Y/%m/%d").strftime("%Y-%m-%d")


def to_marketing_address(address):
    # 'New Rachelburgh-VA-49583' -> 'New Rachelburgh,VA,49583'
    city, state, zip_code = address.rsplit("-", 2)
    return f"{city},{state},{zip_code}"


def parse_user_line(line):
    # One line of users_v.csv -> a user dict
    user_id, name, gender, age, address, date_joined = line.split(",")
    return {
        "user_id": user_id,
        "name": name,
        "gender": gender.capitalize(),
        "age": int(age),
        "address": to_marketing_address(address),
        "state": address.rsplit("-", 2)[1],
        "date_joined": to_iso_date(date_joined),
    }


def format_marketing(user):
    # A user dict -> one ';'-delimited line of marketing_format.csv
    return ";".join([user["user_id"], user["name"], user["gender"], str(user["age"]),
                     user["address"], user["date_joined"]])


def order_user_id(line):
    # One line of orders_v.csv -> its user_id (product_list is quoted and contains commas)
    return next(csv.reader([line]))[1]


def age_band(age):
    # 47 -> '46-60'
    for lo, hi in AGE_BANDS:
        if lo <= age <= hi:
            return f"{lo}-{hi}"
    raise ValueError(f"age {age} is outside the bands")


def utc_timestamp(iso_date):
    # '2019-03-13' -> seconds since 1970-01-01 00:00 UTC (always UTC, whatever your machine's time zone)
    return int(dt.datetime.strptime(iso_date, "%Y-%m-%d").replace(tzinfo=dt.timezone.utc).timestamp())


def utc_date(seconds):
    # seconds since the epoch -> 'YYYY-MM-DD' (UTC)
    return dt.datetime.fromtimestamp(seconds, dt.timezone.utc).strftime("%Y-%m-%d")


def read_output(prefix):
    # All lines written by WriteToText(prefix), across every shard
    lines = []
    for path in sorted(glob.glob(prefix + "-*")):
        with open(path) as f:
            lines.extend(f.read().splitlines())
    return lines


print("Helpers loaded.")

Helpers loaded.


## Task 1: Composite ETL

The retailer's marketing team needs the customer file in the following format, delimited by `;`:

| user_id | name | gender | age | address | date_joined |
| --- | --- | --- | --- | --- | --- |
| `1` | `Anthony Wolf` | `Male` | `73` | `New Rachelburgh,VA,49583` | `2019-03-13` |

### 1a. A composite transform

A composite `PTransform` packages several steps behind one name. It can be reused, and it appears as a single node in the pipeline graph; large pipelines are assembled from composites of this kind.

Complete `ParseUsers`. In `expand`, read the file with `beam.io.ReadFromText`, skipping the header line, and convert each line to a user dict with `parse_user_line`. Later tasks reuse `ParseUsers`.

In [5]:
# 1a ParseUsers: read users_v.csv into a PCollection of user dicts
class ParseUsers(beam.PTransform):

    def __init__(self, path):
        super().__init__()
        self.path = path

    def expand(self, pbegin):
        # TODO: return pbegin | <read self.path, skipping the header> | <parse each line with parse_user_line>
        pass

In [6]:
class ParseUsers(beam.PTransform):

    def __init__(self, filepath):
        self.filepath = filepath

    def expand(self, pbegin):
        return (
            pbegin
            | "ReadUserCSV" >> beam.io.ReadFromText(
                self.filepath,
                skip_header_lines=1
            )
            | "ParseUserRecords" >> beam.Map(parse_user_line)
        )

In [ ]:
p = new_pipeline()

users = (
    p
    | "TestParseUsers" >> ParseUsers(USERS_FILE)
)

# Inspect a few parsed records
users | "PrintSampleUsers" >> beam.Map(print)

p.run().wait_until_finish()

Next, write the marketing file. Apply `ParseUsers(USERS_FILE)`, format each user with `format_marketing`, and write the result with `beam.io.WriteToText` to the prefix `output/marketing_format.csv` with `num_shards=3`. Label every step (`"Name" >> transform`).

Use a `with new_pipeline() as p:` block. The pipeline runs once, when the block exits. Calling `p.run()` after the block would run it a second time, so do not.

In [8]:
# 1a Write marketing_format.csv (3 shards)
MARKETING_PREFIX = os.path.join(OUT_DIR, "marketing_format.csv")


p = new_pipeline()

(
    p
    | "ParseMarketingUsers" >> ParseUsers(USERS_FILE)
    | "FormatMarketingUsers" >> beam.Map(format_marketing)
    | "WriteMarketingCSV" >> beam.io.WriteToText(
        "output/marketing_format",
        num_shards=3
    )
)

result = p.run()
result.wait_until_finish()

print("Marketing extract completed.")

Marketing extract completed.


In [11]:
marketing_lines = read_output("output/marketing_format")

print("Number of records:", len(marketing_lines))
print("\nFirst 3 records:")

for line in marketing_lines[:3]:
    print(line)

Number of records: 2357

First 3 records:
3;Cody Shaw;Male;75;North Anne,SC,53799;2004-05-29
6;Sierra Andrews;Female;21;Ryanville,MI,69690;2007-05-25
9;Ronald Meyer;Male;41;North Cherylhaven,NJ,04197;2015-11-14


In [10]:
import hashlib

sorted_lines = sorted(marketing_lines)

digest = hashlib.sha256(
    "\n".join(sorted_lines).encode("utf-8")
).hexdigest()

print("Q01:", len(marketing_lines), digest[:12])

Q01: 2357 d95da6a2c7c2


In [14]:
shard_files = sorted(
    glob.glob("output/marketing_format-*")
)

print("Number of shard files:", len(shard_files))

for file in shard_files:
    print(file)

Number of shard files: 3
output/marketing_format-00000-of-00003
output/marketing_format-00001-of-00003
output/marketing_format-00002-of-00003


The check below reads every shard, sorts the lines and computes a fingerprint, so that the result does not depend on how the lines were distributed across shards.

* 1a. Report the number of lines written and the first 12 characters of the sorted-lines SHA-256.

In [20]:
# 1a check: line count and fingerprint of your output
lines = read_output(MARKETING_PREFIX)
assert lines, "No output found. Complete ParseUsers and the write cell above, then re-run this cell."
fingerprint = hashlib.sha256(("\n".join(sorted(lines)) + "\n").encode()).hexdigest()
print(f"Lines written: {len(lines):,}")
print(f"Sorted-lines SHA-256 (first 12): {fingerprint[:12]}")
print("First 3 lines (sorted):", *sorted(lines)[:3], sep="\n  ")

AssertionError: No output found. Complete ParseUsers and the write cell above, then re-run this cell.

### 1b. Sharded output

Inspect the files that `WriteToText` produced.

In [15]:
# 1b Inspect the shards
for path in sorted(glob.glob(MARKETING_PREFIX + "-*")):
    with open(path) as f:
        shard = f.read().splitlines()
    print(f"{os.path.basename(path)}: {len(shard):,} lines, first line: {shard[0] if shard else '-'}")

* 1b. (i) How many files did the pipeline write? (ii) Is the order of the output lines guaranteed to match the input file? (iii) Why does Beam write several files rather than one?

### 1c. Unit-testing a transform

A `TestPipeline` with `assert_that` checks a transform's output inside the pipeline. Tests of this kind keep a large pipeline correct as it grows.

The cell below writes a small fixture: the header and the first three users. Complete the test that follows it: write out by hand the three user dicts that `ParseUsers` should produce from the fixture, and assert that it produces exactly those.

In [25]:
# 1c Fixture: header + first 3 users
FIXTURE_FILE = os.path.join(OUT_DIR, "users_fixture.csv")
with open(USERS_FILE) as src, open(FIXTURE_FILE, "w") as dst:
    dst.writelines(next(src) for _ in range(4))
!cat {FIXTURE_FILE}

with open(FIXTURE_FILE, "r") as f:
    print(f.read())

user_id,name,gender,age,address,date_joined
1,Anthony Wolf,male,73,New Rachelburgh-VA-49583,2019/03/13
2,James Armstrong,male,56,North Jillianfort-UT-86454,2020/11/06
3,Cody Shaw,male,75,North Anne-SC-53799,2004/05/29
user_id,name,gender,age,address,date_joined
1,Anthony Wolf,male,73,New Rachelburgh-VA-49583,2019/03/13
2,James Armstrong,male,56,North Jillianfort-UT-86454,2020/11/06
3,Cody Shaw,male,75,North Anne-SC-53799,2004/05/29



In [28]:
expected = [
    {
        "user_id": "1",
        "name": "Anthony Wolf",
        "gender": "Male",
        "age": 73,
        "address": "New Rachelburgh,VA,49583",
        "state": "VA",
        "date_joined": "2019-03-13"
    },
    {
        "user_id": "2",
        "name": "James Armstrong",
        "gender": "Male",
        "age": 56,
        "address": "North Jillianfort,UT,86454",
        "state": "UT",
        "date_joined": "2020-11-06"
    },
    {
        "user_id": "3",
        "name": "Cody Shaw",
        "gender": "Male",
        "age": 75,
        "address": "North Anne,SC,53799",
        "state": "SC",
        "date_joined": "2004-05-29"
    }
]

print("Expected records:", len(expected))

Expected records: 3


In [29]:
with TestPipeline(options=PipelineOptions(flags=[])) as p:
    actual = p | "ParseFixtureUsers" >> ParseUsers(FIXTURE_FILE)

    assert_that(actual, equal_to(expected))

print("ParseUsers test passed!")

ParseUsers test passed!


A test is only useful if it fails when the output is wrong. Break it deliberately: copy `expected`, change user 1's age to 74, and run the same test against the copy. The `try` block keeps the notebook running when the test fails, and prints the assertion lines of the error message.

* 1c. Which assertion exception does the error message name?

In [32]:
import copy

broken = copy.deepcopy(expected)
broken[0]["age"] = 74

try:
    with TestPipeline(options=PipelineOptions(flags=[])) as p:
        actual = p | "ParseBrokenFixture" >> ParseUsers(FIXTURE_FILE)

        assert_that(actual, equal_to(broken))

except Exception as error:
    print("Exception type:", type(error).__name__)
    print("Exception message:")
    print(error)

ERROR:apache_beam.runners.common:Failed assert: [{'user_id': '1', 'name': 'Anthony Wolf', 'gender': 'Male', 'age': 74, 'address': 'New Rachelburgh,VA,49583', 'state': 'VA', 'date_joined': '2019-03-13'}, {'user_id': '2', 'name': 'James Armstrong', 'gender': 'Male', 'age': 56, 'address': 'North Jillianfort,UT,86454', 'state': 'UT', 'date_joined': '2020-11-06'}, {'user_id': '3', 'name': 'Cody Shaw', 'gender': 'Male', 'age': 75, 'address': 'North Anne,SC,53799', 'state': 'SC', 'date_joined': '2004-05-29'}] == [{'user_id': '1', 'name': 'Anthony Wolf', 'gender': 'Male', 'age': 73, 'address': 'New Rachelburgh,VA,49583', 'state': 'VA', 'date_joined': '2019-03-13'}, {'user_id': '2', 'name': 'James Armstrong', 'gender': 'Male', 'age': 56, 'address': 'North Jillianfort,UT,86454', 'state': 'UT', 'date_joined': '2020-11-06'}, {'user_id': '3', 'name': 'Cody Shaw', 'gender': 'Male', 'age': 75, 'address': 'North Anne,SC,53799', 'state': 'SC', 'date_joined': '2004-05-29'}], unexpected elements [{'user_

Exception type: BeamAssertException
Exception message:
Failed assert: [{'user_id': '1', 'name': 'Anthony Wolf', 'gender': 'Male', 'age': 74, 'address': 'New Rachelburgh,VA,49583', 'state': 'VA', 'date_joined': '2019-03-13'}, {'user_id': '2', 'name': 'James Armstrong', 'gender': 'Male', 'age': 56, 'address': 'North Jillianfort,UT,86454', 'state': 'UT', 'date_joined': '2020-11-06'}, {'user_id': '3', 'name': 'Cody Shaw', 'gender': 'Male', 'age': 75, 'address': 'North Anne,SC,53799', 'state': 'SC', 'date_joined': '2004-05-29'}] == [{'user_id': '1', 'name': 'Anthony Wolf', 'gender': 'Male', 'age': 73, 'address': 'New Rachelburgh,VA,49583', 'state': 'VA', 'date_joined': '2019-03-13'}, {'user_id': '2', 'name': 'James Armstrong', 'gender': 'Male', 'age': 56, 'address': 'North Jillianfort,UT,86454', 'state': 'UT', 'date_joined': '2020-11-06'}, {'user_id': '3', 'name': 'Cody Shaw', 'gender': 'Male', 'age': 75, 'address': 'North Anne,SC,53799', 'state': 'SC', 'date_joined': '2004-05-29'}], unexpe

## Task 2: Combining

Beam has three combining patterns that this task uses:

| Result needed | Transform |
| :--- | :--- |
| one value per key | `beam.CombinePerKey(...)`, or `beam.combiners.Count.PerKey()` / `Count.PerElement()` |
| one value for the whole collection | `beam.CombineGlobally(...)`, or `beam.combiners.Count.Globally()` |
| the top *n* values within each key | `beam.combiners.Top.PerKey(n)` |

### 2a. Gender breakdown

Count customers per gender and compute each gender's share of all customers as a percentage (2 decimal places). Both must be computed inside the pipeline: obtain the total with a global count, and pass it as a side input (`beam.pvalue.AsSingleton(total)`) to the step that computes the percentages. Dividing in plain Python after the pipeline has run does not satisfy the task.

* 2a. Report the number of customers and the percentage for `Female` and for `Male`.

In [ ]:
# 2a Gender counts and percentages
# TODO

### 2b. Top states per gender

For each gender, find the two states with the most customers. Count per (gender, state), re-key so that gender is the key, then apply `beam.combiners.Top.PerKey(2)`. `Top` compares whole values, so place the count first in each value, for example `(count, state)`.

* 2b. Report the top two states, with their counts, for each gender.

In [ ]:
# 2b Top 2 states per gender
# TODO

## Task 3: Batch and windowed counts as one model

A window groups elements by time. Beam's `FixedWindows(size)` divides time into intervals of equal length, counted from the Unix epoch (1970-01-01 00:00 UTC). An element with timestamp `t` belongs to the window that starts at `t - t % size`.

This task counts customer sign-ups per 30-day window twice, first as an ordinary batch group-by and then with Beam's windowing, and compares the results.

### 3a. Batch

Key each user by the start date of the 30-day window that contains their `date_joined`:

* obtain the timestamp with `utc_timestamp(user["date_joined"])`;
* compute the window start, `t - t % (WINDOW_DAYS * DAY)`;
* convert it back to a date with `utc_date(...)`.

Count per key and write lines of the form `YYYY-MM-DD,count` to `output/t3_batch`.

Build this pipeline without a `with` block (`p3 = new_pipeline()`) and do not run it yet. In 3c its graph is drawn first and the pipeline is then run.

In [ ]:
# 3a Sign-ups per 30-day window, batch
BATCH_PREFIX = os.path.join(OUT_DIR, "t3_batch")
p3 = new_pipeline()
# TODO: p3 | ParseUsers(USERS_FILE) | ... | beam.io.WriteToText(BATCH_PREFIX)

### 3b. Windowed

Now let Beam do the grouping. Assign each user an event timestamp taken from `date_joined` (`TimestampedValue`), window the users into 30-day fixed intervals (`FixedWindows`), and count sign-ups globally per window, without default values for empty windows (`CombineGlobally` with `CountCombineFn`). No key is needed, because the window acts as the key.

Use the provided `EmitWindowStart` to turn each count into a `YYYY-MM-DD,count` line, and write the lines to `output/t3_windowed`.

In [ ]:
# 3b Sign-ups per 30-day window, windowed
WINDOWED_PREFIX = os.path.join(OUT_DIR, "t3_windowed")


class EmitWindowStart(beam.DoFn):
    # A count -> 'YYYY-MM-DD,count', where the date is the start of the count's window
    def process(self, count, window=beam.DoFn.WindowParam):
        yield f"{utc_date(int(window.start))},{count}"


# TODO: with new_pipeline() as p: ... | beam.ParDo(EmitWindowStart()) | beam.io.WriteToText(WINDOWED_PREFIX)

### 3c. The pipeline graph

A Beam pipeline is a directed acyclic graph (DAG) of transforms. The cell below draws the 3a pipeline, with one box per labelled transform and one circle per `PCollection`, and then runs it once.

In [ ]:
# 3c Draw the graph of p3, then run it once
from apache_beam.runners.interactive.display import pipeline_graph

dot = pipeline_graph.PipelineGraph(p3).get_dot()
try:
    import graphviz
    display(graphviz.Source(dot))
except Exception:
    print(dot)   # Graphviz is not available here: the graph as text

p3.run().wait_until_finish()   # no `with` block, so the pipeline is run explicitly, exactly once
batch_counts = dict(line.split(",") for line in read_output(BATCH_PREFIX))

In [ ]:
# 3 Compare batch and windowed
windowed_counts = dict(line.split(",") for line in read_output(WINDOWED_PREFIX))
assert batch_counts and windowed_counts, "No output found. Complete 3a and 3b, then re-run 3c and this cell."
print(f"Windows with sign-ups: batch {len(batch_counts)}, windowed {len(windowed_counts)}")
print("Batch == windowed:", batch_counts == windowed_counts)

* 3 (i). How many 30-day windows contain at least one sign-up?
* 3 (ii). The busiest single sign-up day was 2018-10-17. On what date does the 30-day window containing it start, and how many sign-ups does that window hold?
* 3 (iii). Why does that window start on that date rather than on 1 October?
* 3c (i). Which transform in the graph requires a shuffle, that is, moving data between workers so that all elements with the same key end up together? Why does no `Map` step require one?
* 3c (ii). Is the data in this task bounded or unbounded? What does that imply for the watermark, and why are the batch and windowed counts equal?

## Task 4 (Track A): Joining customers and orders

`CoGroupByKey` joins several keyed `PCollection`s. For each key it returns a dict of lists, one list per tagged input, for example `('17', {'user': [<user dict>], 'orders': [1, 1, 1, ...]})`.

### 4a. Order profiles

1. Key users by `user_id`, and key each order line by `order_user_id(line)` with the value 1. Skip the header of `orders_v.csv`.
2. Join the two collections on `user_id` with `beam.CoGroupByKey()`.
3. For each customer, emit `(user, number_of_orders)`.
4. Compute the average number of orders per customer (2 decimal places), first by gender and then by age band (`age_band(user["age"])`). `beam.combiners.Mean.PerKey()` is useful here.

The orders file has more than 800,000 lines.

* 4a (i). Report the average number of orders per customer for `Female` and for `Male`.
* 4a (ii). Report the average number of orders per customer for each age band: `18-30`, `31-45`, `46-60`, `61-80`.

In [ ]:
# 4a Join users and orders; average orders per customer by gender and by age band
# TODO

### 4b. A live dashboard (design decision)

Task 4b (at most 200 words). The batch and windowed sign-up counts in Task 3 agreed. The retailer now wants a live daily sign-up dashboard. Sign-up events reach the pipeline late and out of order, because the mobile app caches them when connectivity is poor. Choose one option:

* (A) show each day only once the watermark has passed it: complete, but late;
* (B) show early results, then update them as late data arrives;
* (C) group by arrival (processing) time instead: fast, but some sign-ups are counted on the wrong day.

Writing for a marketing manager who checks the dashboard every morning:

1. Justify your choice.
2. Explain why the Task 3 counts agreed, and whether your dashboard's figures would match an end-of-month batch report.
3. State the main risk of your choice and how you would reduce it.

Draft your answer below and submit it as your handout instructs.

Your answer:

## Task 5 (Track B): Late data

In a stream, events arrive out of order. The watermark is the pipeline's estimate that all events up to time *t* have arrived. When the watermark passes the end of a window, the window's on-time result fires, and events for that window that arrive afterwards are late.

* Allowed lateness sets how long after the end of a window late events are still accepted, and so how long the window's state is kept.
* A late trigger (`AfterCount(1)`) fires an additional pane for each accepted late event.
* The accumulation mode sets whether a late pane reports the window's running total (`ACCUMULATING`) or only the new elements (`DISCARDING`).

`TestStream` replays a scripted stream with explicit watermark advances, so the results are identical on every run. Event times are given in days (timestamp = day × `DAY`):

| Step | Action |
| :--- | :--- |
| 1 | add events at days 3, 12, 25, 33 |
| 2 | advance the watermark to day 31 |
| 3 | add events at days 28, 35 |
| 4 | advance the watermark to day 40 |
| 5 | add events at days 5, 38 |
| 6 | advance the watermark to day 61, then to infinity |

### 5a. The late-data pipeline

Complete `run_late_data`. Window into `FixedWindows(WINDOW_DAYS * DAY)` with:

* the trigger `AfterWatermark(late=AfterCount(1))`;
* the given allowed lateness, converted from days to seconds;
* the given accumulation mode.

Then count per key with `beam.combiners.Count.PerKey()`. `RecordPane` prints one line for each pane that fires.

In [ ]:
# 5a Provided: the scripted stream, and a DoFn that prints every pane
from apache_beam.testing.test_stream import TestStream
from apache_beam.transforms import trigger, window

STREAM_SCRIPT = [("events", [3, 12, 25, 33]), ("watermark", 31),
                 ("events", [28, 35]), ("watermark", 40),
                 ("events", [5, 38]), ("watermark", 61)]


def make_test_stream():
    stream = TestStream()
    for kind, value in STREAM_SCRIPT:
        if kind == "events":
            stream = stream.add_elements([window.TimestampedValue(("till", d), d * DAY) for d in value])
        else:
            stream = stream.advance_watermark_to(value * DAY)
    return stream.advance_watermark_to_infinity()


class RecordPane(beam.DoFn):
    def process(self, key_count, w=beam.DoFn.WindowParam, pane=beam.DoFn.PaneInfoParam):
        timing = {1: "ON_TIME", 2: "LATE"}.get(pane.timing, pane.timing)
        print(f"  window [{int(w.start) // DAY:>2}, {int(w.end) // DAY:>2}) pane {pane.index} {timing:<7} count = {key_count[1]}")

In [ ]:
# 5a Late-data pipeline
def run_late_data(allowed_lateness_days, accumulation_mode):
    with new_pipeline(streaming=True) as p:
        (p
         | "Stream" >> make_test_stream()
         # TODO: | "Window" >> beam.WindowInto(...)
         # TODO: | "Count" >> ...
         | "Record" >> beam.ParDo(RecordPane()))

In [ ]:
# 5a Run all four configurations
for lateness in (0, 7):
    for mode in ("ACCUMULATING", "DISCARDING"):
        print(f"allowed lateness = {lateness} days, {mode}:")
        run_late_data(lateness, getattr(trigger.AccumulationMode, mode))

The stream contains 8 events. Answer from your output:

* 5a (i). How many events are dropped with an allowed lateness of 0 days?
* 5a (ii). How many events are dropped with an allowed lateness of 7 days?
* 5a (iii). With 7 days of allowed lateness, how many panes fire for the window [0, 30)?
* 5a (iv). With 7 days of allowed lateness, how many elements does the last pane of the window [0, 30) report in `ACCUMULATING` mode, and in `DISCARDING` mode?

Before answering, use the pane output to work out what happened to the day-28 and day-5 events under each lateness setting, and why.

### 5b. A lateness policy (design decision)

Task 5b (at most 200 words). A point-of-sale system issues a discount voucher when product X is in a basket. Tills that lose connectivity upload their events up to N days late. Take L, the last digit of your student number: L = 0–3 gives N = 2 days, L = 4–6 gives N = 5 days, L = 7–9 gives N = 10 days. Begin your answer with "L = …, N = …".

Writing for the retail operations lead:

1. Specify the allowed lateness, the trigger and the accumulation mode for this pipeline.
2. Justify the trade-off between a missed voucher, a voucher issued wrongly or late, and the state cost of keeping windows open, with reference to your Task 5a results.
3. State the main risk of your design and how you would reduce it.

Draft your answer below and submit it as your handout instructs.

Your answer:

## Evidence and version control

### 1. Export the evidence PDF first

The executed PDF is the record of your work. Every code cell and its output, including the check cells, the results, the pipeline graph and the pane output, must be visible.

1. In Google Colab, select File > Print > Save as PDF.
2. Confirm that the outputs and the graph are readable and not truncated.
3. Upload the PDF to STEMLearn as your handout instructs.

### 2. Clear outputs, then commit

1. Select Edit > Clear all outputs, then save the notebook into your repository.
2. Keep data and outputs out of Git: add `*.csv`, `data/` and `output/` to your `.gitignore` before committing.

Repository, branch, merge-request and submission details are in your course handout.

In [ ]:
# Before committing, check locally: git status && git log -n 5 --oneline
# Confirm that no data or output file is tracked and that the committed notebook has no outputs.
print("Export the evidence PDF and clear all outputs before committing.")